In [0]:
# ============================================================

# GOLD – dim_project

# Grain: (source_user_id, project)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_TICKETS = "workspace.slainte_silver.easyvista_tickets_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_PROJECT = f"{GOLD_DB}.dim_project"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_tickets = spark.table(SILVER_TICKETS)

# ---------------- BUILD DIM ----------------

df_project_base = (

    df_tickets

    .select(

        F.col("source_user_id"),

        F.col("project")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull()

    )

    .distinct()

)

window = Window.orderBy("source_user_id", "project")

df_dim_project = (

    df_project_base

    .withColumn("project_id", F.row_number().over(window))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "project_id",

        "source_user_id",

        "project"

    )

)

# ---------------- WRITE ----------------

df_dim_project.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_PROJECT)

print("✅ dim_project created successfully")

display(spark.table(DIM_PROJECT).orderBy("source_user_id", "project"))
 